<a href="https://colab.research.google.com/github/srijansalian109-del/ScamBait1/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [19]:

!pip install google-genai --quiet
print("Done.")

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main.py", line 47, in main
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main_parser.py", line 11, in <module>
    from pip._internal.cli import cmdoptions
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/cmdoptions.py", line 28, in <module>
    from pip._internal.cli.parser import ConfigOptionParser
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/parser.py", line 20, in <module>
    from pip._internal.configuration import Configuration, ConfigurationError
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/configuration.p

In [20]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
    print("API key set.")


Paste your Gemini API key: ··········
API key set.


In [21]:
DOCUMENTS = {
    "neer_dosa": (
        "1 cup raw rice (sona masuri or any raw rice). "
        "1/2 cup freshly grated coconut. "
        "1 tsp salt. "
        "1.5 to 2 cups water as needed. "
        "1 tsp oil for greasing the pan. "
        "Soak the raw rice in water for 4 hours then drain. "
        "Grind the rice with the grated coconut and water into a very smooth and thin batter. The batter should be watery like buttermilk. Add salt and mix. "
        "No fermentation is needed. The batter can be used immediately. "
        "Heat a non-stick tawa on medium heat and grease it lightly with oil. "
        "Pour a ladle of batter from a height and swirl the pan to spread it into a thin lacy circle. Do not flip the dosa. "
        "Cover and cook for 1-2 minutes until the edges lift and the dosa turns soft and cooked. "
        "Fold into quarters and serve hot with coconut chutney or chicken curry."
    ),
    "golibaje": (
        "1 cup maida (all-purpose flour). "
        "1/2 cup curd. "
        "2 tablespoons rice flour. "
        "1 tablespoon sugar. "
        "1/2 teaspoon cumin seeds. "
        "1/2 teaspoon baking soda. "
        "1 green chilli finely chopped. "
        "1 tablespoon chopped coriander. "
        "1/2 teaspoon grated ginger. "
        "Salt to taste. "
        "Water as required. "
        "Oil for deep frying. "
        "Mix maida rice flour curd sugar cumin seeds baking soda and salt in a bowl. "
        "Add chopped green chilli ginger and coriander. "
        "Add water little by little and mix to make a thick sticky batter. "
        "Cover the batter and let it rest for 2 to 3 hours. "
        "Heat oil in a deep pan over medium heat. "
        "Wet your hands, take small portions of the batter and gently drop them into the hot oil. "
        "Deep fry until the golibaje become golden brown and crispy. "
        "Drain on a paper towel and serve hot with coconut chutney."
    ),
    "mangalore_buns": (
        "2 cups maida (all-purpose flour). "
        "2 ripe bananas. "
        "1/4 cup sugar. "
        "2 tablespoons curd. "
        "1/2 teaspoon cumin seeds. "
        "1/2 teaspoon baking soda. "
        "A pinch of salt. "
        "1 tablespoon ghee. "
        "Oil for deep frying. "
        "Mash the ripe bananas in a bowl. "
        "Add sugar curd ghee cumin seeds baking soda and salt. "
        "Mix well and gradually add maida to form a soft dough. "
        "Knead the dough until it becomes smooth and slightly sticky. "
        "Cover and let the dough rest for 6 to 8 hours or overnight. "
        "Divide the dough into small balls. "
        "Roll each ball into a small thick circle using a little oil. "
        "Heat oil in a deep pan over medium heat. "
        "Deep fry the buns until they puff up and turn golden brown on both sides. "
        "Drain excess oil and serve warm with coconut chutney or sambar."
    ),
}

In [22]:
import re

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "with", "is",
    "are", "how", "what", "which", "do", "does", "can", "i", "we", "our", "it",
    "long", "much", "many", "need", "needed", "make", "ingredients", "recipe",
}


def search_documents(query: str) -> str:
    """Search the cookbook for recipes relevant to the query."""
    words = {w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOPWORDS}
    results = []

    for name, content in DOCUMENTS.items():
        title_words = set(name.replace("_", " ").split())
        text_words = set(re.findall(r"[a-z]+", content.lower()))
        score = sum(3 for w in words if w in title_words)   # title match counts more
        score += sum(1 for w in words if w in text_words)   # whole-word content match
        if score > 0:
            results.append((score, name, content))

    results.sort(key=lambda r: r[0], reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    return "\n\n".join(f"[Source: {n}]\n{c}" for _, n, c in results[:2])


print(search_documents("golibaje"))

[Source: golibaje]
1 cup maida (all-purpose flour). 1/2 cup curd. 2 tablespoons rice flour. 1 tablespoon sugar. 1/2 teaspoon cumin seeds. 1/2 teaspoon baking soda. 1 green chilli finely chopped. 1 tablespoon chopped coriander. 1/2 teaspoon grated ginger. Salt to taste. Water as required. Oil for deep frying. Mix maida rice flour curd sugar cumin seeds baking soda and salt in a bowl. Add chopped green chilli ginger and coriander. Add water little by little and mix to make a thick sticky batter. Cover the batter and let it rest for 2 to 3 hours. Heat oil in a deep pan over medium heat. Wet your hands, take small portions of the batter and gently drop them into the hot oil. Deep fry until the golibaje become golden brown and crispy. Drain on a paper towel and serve hot with coconut chutney.


In [23]:
import time
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# NOTE: gemini-2.5-flash returns 404 for new users; gemini-3.8-flash is the current model.
# To list models available to your key:
#   for m in client.models.list(): print(m.name)
MODEL_NAME = "gemini-3.8-flash"

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the Coastal Karnataka cookbook for relevant recipes.",
            parameters={
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    system_instruction=(
        "You are a chef training at a coastal Karnataka restaurant in Mangaluru. "
        "Use search_documents for questions about ingredients, recipes, cooking "
        "times, and preparation methods. Answer only using the retrieved documents. "
        "If the documents do not contain the answer, say so clearly."
    ),
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

TOOLS = {"search_documents": search_documents}


In [24]:
def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=config
            )
        except Exception as e:
            error = str(e)
            if "429" in error:
                wait = 20 * (attempt + 1)   # free tier: 10 requests/minute
            elif "503" in error:
                wait = 5 * (attempt + 1)
            else:
                raise
            if attempt < max_retries - 1:
                print(f"Temporary API error. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        if not response.candidates or not response.candidates[0].content:
            print("No response returned by Gemini.")
            return ""

        model_content = response.candidates[0].content
        contents.append(model_content)   # keep the model turn unmodified

        calls = [p.function_call for p in (model_content.parts or []) if p.function_call]

        if not calls:
            answer = response.text or "No text answer returned."
            print(f"\nFinal answer:\n{answer}")
            return answer

        result_parts = []
        for call in calls:
            func = TOOLS.get(call.name)
            if func is None:
                result = "Unknown tool."
            else:
                args = dict(call.args)
                print(f"Step {step}: {call.name}({args})")
                result = func(**args)
                print(f"  Result preview: {result[:100]}...")

            result_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        id=call.id,          # may be None; that is fine
                        name=call.name,
                        response={"result": result},
                    )
                )
            )

        contents.append(types.Content(role="user", parts=result_parts))

    print("Stopped: maximum steps reached.")
    return ""


In [25]:
questions = [
    "What are the ingredients needed to make golibaje?",
    "How long should the mangalore buns dough rest?",
    "Does the neer dosa batter need fermentation?",
    "What is the most famous street food in Mumbai?",   # not in documents
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(20)   # stay inside the free-tier rate limit
    print("\n" + "=" * 60)
    run_agent(q)


Question: What are the ingredients needed to make golibaje?
--------------------------------------------------
Step 1: search_documents({'query': 'golibaje'})
  Result preview: [Source: golibaje]
1 cup maida (all-purpose flour). 1/2 cup curd. 2 tablespoons rice flour. 1 tables...
Temporary API error. Retrying in 5s...
Temporary API error. Retrying in 10s...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)